# Benchmark findings

These studies evaluate fitted models or analyst workflows against planted
structure, held-out error, or competing libraries. The summaries below
report the findings and subsequent changes recorded by those runs.

Results belong to the datasets, settings, and code versions in their
artifacts under `docs/benchmarks/` and `docs/book/benchmarks/`. Refer to the
linked study and current issue status before treating an old result as a
measurement of the current release.

## Capacity selection and transfer

**Fixed and selected capacity.** The frozen predictive study
(the published artifact `docs/benchmarks/competitive_shootout_report.md`) measured the additive model a median 46.5% behind the best
competing library per panel, and measured the cost of the standard
remedy: an always-on joint tree turned the cancer panel from 11% ahead of
the best competitor to 109% behind. The preferred learner capacity differed across datasets.
`select_capacity` (#479) scores a ladder of nested specifications—base,
screened tensors, and one joint tree—out of fold on one shared fold partition.
It keeps the smallest rung within two standard errors of the best. A rerun
of the frozen panels with the chooser (`benches/probe_chooser_shootout.py`,
October 2026) reported these changes: cancer fell from +104.0%
excess to +7.6%, the smooth panels were kept (`smooth_sparse` -0.9%,
`correlated_proxies` -3.5%, both ahead of the best competitor) and
`concrete` improved from +19.6% to +12.7%. The same rerun traced three
remaining gaps, filed as #493: the tree rung reads only numeric columns,
so the bike panels' hour-by-working-day interaction -- a categorical --
is invisible to it and the additive-level excess returns; a screened pair
on a two-valued column crashes the tensor construction; and one shared
step budget with a fixed tree depth undertrains richer rungs on
`joint_boundary` and `diamonds`.

**`effect_model` could not represent a switching conditional effect.**
The study's causal arm planted an effect that switches on inside the
range of a modifier (`1 + 1.5 sign(x0 x1)`) and measured PEHE 1.548 with
no available alternative: both candidate dictionaries were additive in the
modifiers. The fix added a third candidate -- a depth-3 `btree` over all
modifiers pooled with `selection=sse`, adopted only when its
cross-validated R-loss is at least 5% below the best additive candidate.
On seven confirmation draws held out from the three draws that chose the
depth, the panel improved to PEHE 1.255 and selected the joint dictionary
in 7 of 7 draws, while the three additive panels selected exactly the
dictionary they had before in all 21 draws. The limitation recorded with
the fix: a joint tree fixes the estimation failure without producing a
readable one-curve-per-modifier decomposition.

**Transfer from CatBoost summaries.** Two probes
transplanted a tuned CatBoost teacher's knowledge into an mboostrs
specification -- its native importances and top interaction pair, and
spline degrees of freedom chosen to reproduce its partial-dependence
curves to 2% residual. Both scored worse than the plain tuned hybrid that
never looked at the teacher (protein 21.2 against 18.9 signal MSE;
superconductivity 178 against 140; housing 0.0595 against 0.0555 held-out
MSE). The probes do not establish which part of the transfer caused the higher
error. The recorded artifacts (`docs/benchmarks/distillation_probe.json`,
`shape_transfer_probe.json`) support evaluating the transferred
specification on held-out data rather than assuming that the teacher's
feature summaries will improve it.

## PM2.5 forecasting

The competition page is
[](../benchmarks/pm25_forecast_competition); its artifacts live under
`docs/book/benchmarks/pm25_forecast/`.

**Reference-model comparison.** The mboostrs
analyst tuned one additive formula and finished 7% behind CatBoost on
2014 RMSE of log PM2.5 (0.6615 against 0.6181) without ever asking how
much a flexible model would still gain. The workflow did not include a reference-model comparison.
Subsequent work added
`reference_gap` (#473), SKILL step 5, and the `modeling-tactics` guide
topic (#469) -- and the follow-up investigation closed 58% of the gap
with lagged weather, tensor interactions and wind-direction `by=` terms,
each chosen on the validation year: 2014 RMSE 0.636, and on the
validation year the median pinball beat CatBoost (21.21 against 21.27).

**Response transformation.** The response is
positive with a 99th percentile near 400 µg/m³ against a median near 70;
fitting the raw scale cost the three-quantile mean pinball 21.2, and
`log1p` brought it to 14.0. Quantiles are unchanged by a monotone
transform, so the fitted log-scale quantiles map back exactly. The same
lesson applies to covariates: an average-occupancy covariate met a test
row at 1,243 against a training maximum of 6.57, and the spline's
straight-line extension predicted 338 for a truth near 0. The fixes are
the log-scale guidance and the `extrapolation=` policy with per-term
out-of-range warnings (#454).

**Capacity selection on rolling folds.** A 2026 rerun ran the
chooser from the analyst's exact submitted formula with rolling-origin
folds: all three rungs scored within noise (0.2511, 0.2507, 0.2518), the
chooser kept the base, and the 2014 RMSE was 0.6604 against the 0.6615
submission. The manual loop's gains had come from new features -- lags --
not from capacity, and the rolling-fold comparison did not support adding a richer learner
specification. Two open items follow
from this battery: the 80% interval covers 0.74 overall (0.68 in winter)
for every library, and the chooser's two-standard-error tolerance on
five rolling folds demands roughly a 14% improvement to adopt, which is
a large threshold relative to the improvements in this comparison.

## Claim-cost discovery

The battery page is [Claim-cost discovery](../benchmarks/claims_discovery).

**Failed calls.** Refits, confidence intervals
and conformal calibration raised on string category labels, an
empty-selection formula produced an error instead of the intercept-only
model, and the guide told agents that no intervals existed. The automated-mode scorecard put
mboostrs last at 2.95 against statsmodels' 3.48, with the profile showing
the library ahead on completed runs. The fixes -- string factors through
every refit and interval path, the empty-selection rule, the topic-based
agent guide with its worked examples -- moved mboostrs to 3.8 against
statsmodels' 3.5 on the rerun.

**Missing covariates.**
Housing age was missing on 4-16% of rows and roof class was assigned
partly from age, so the median fill let roof-class coefficients absorb
the age effect. The fix, `na_action="impute_model"`, fills each gap from
the other columns with a penalized additive regression in the core and
keeps the missingness indicator; roof-class interval coverage rose from
0.30 to 0.80. The measurement also rejected a fix: splitting each smooth
on a filled column into observed and filled parts lowered housing-age
coverage from 0.90 to 0.60, and the change was closed unmerged (#467).

**Regression checks.** After the
rerun's pipeline changes, interaction recall fell from 0.81 to 0.38 and
effect recall from 0.58 to 0.33 (#415); the fix (#436) restored the
screen and removed a crash on the same path. These recovery measures identified a regression that the existing test
suite had not detected.

For every library in this comparison, a narrow spatial effect (a bump of 0.4 degrees) is not resolvable
by a 12-by-12 basis, and all libraries' 90% intervals covered it on at
most 1 of 8 replicates. Raising the basis size for narrow features is the
recorded remedy, and `mb.gam` refits fast enough to try several.

## Causal-effect estimation

The battery page is [Causal discovery](../benchmarks/causal_discovery).

**Separate per-arm outcome models extrapolated into thin overlap.** The
first pipeline's doubly-robust fits blew up on rows with cross-fitted
propensities near 0 and 1 -- mean absolute error on the average effect
reached \$5.57 million, and the no-effect data drew false claims. The
recorded changes included a two-part outcome model
and skewed covariates on the log scale (#434), the overlap-weighted
target (#426), boundary knots and log-scale modifiers in `effect_model`
(#428), and spec defaults carried into the estimator (#432). Measured on
the gap study: average-effect error fell to \$1,019, the no-effect mean
error to \$28, ranking regret from 0.80 to 0.47 and then 0.34 for the
automatic ranking, with the automatic mode 3.5 times faster. Across four
fix rounds the weighted scorecard moved from 3.03 to 3.21 (automated) and
3.06 to 3.26 (semi-automated), with the fourth run reproducing exactly
and holding the best effect-query coverage, 0.91.

Open items this battery still carries: statsmodels' 3.53 remains ahead on
the weighted profile; average-effect interval coverage fell from 7 of 8
replicates to 5 of 8 in the last rerun (#443); the adjustment-set grade
stayed at 2 (#478); and the friction logs from reruns 5 and 6 are filed
as #484 and #487.

## Discontinuity screening

**Diagonal regime boundaries.** The second
agentic battery planted a +2.0 jump across the boundary
`s1 = 0.45 + 0.35 s0` at n=800 with noise sd 0.5: smooth fits absorb most
of a step, axis-aligned step candidates read "no gain" in
cross-validation, and nothing in the workflow pointed at diagonal
boundaries. An analyst using xgboost found the boundary by grid-scanning
diagonal indicators. The fix is `screen_discontinuities` (#488, with the
Rust kernel `disc_scan`): axis-aligned and diagonal boundary scans that ranks candidates on one half of the rows and estimates the reported
z statistic on the other half with a
type statistic that separates a jump from a steep smooth crossing.
Measured on the planted panel: the boundary recovered at 99.2% row
agreement (jump 1.965 of the planted 2.0, held-out z 16.1), and the
boundary's step indicator beats the additive baseline by 0.126
out-of-fold MSE (5.2 paired standard errors) while the `btree(pair)`
candidate, the default when the boundary is unknown, does not improve on the
additive baseline.
A 200-replicate null calibration of the shipped gate: the top row flags
on 1.0% of no-jump screens, the mean absolute z is 0.81 against the
theoretical 0.80, and steep smooth crossings flag on the raw response
but type as jumps on only 8% of screens.

## Double-penalty REML

The pilot (#462) asked whether explicit null-space shrinkage can separate
term suppression from smoothing, using the existing Rust REML engine on
four Gaussian scenarios at 20 seeds (PR #482). The reported result depended on the scenario: the double penalty cuts signal MSE 26% on sparse signals
and 57% with a correlated substitute, but raises it 30% on dense signals
and fails the prespecified across-scenario gate, so the default stayed
and the recorded recommendation is training-only selection among
boosting, joint REML and double-penalty REML on common inner folds. In the same table, all-term joint REML had lower error than tuned boosting
on every non-null scenario at n=600, p=8, and took less time than boosting's
tuning procedure. Boosting performed best on the null scenario.

## Planted-insight recovery

The no-skill cold-agent procedure recovers 10 of 10 planted insights
with every direction correct and an uncertainty score of 0.56, its
recorded baseline (`python -m eval_insights` run from `benches/`, deterministic; #290).
The October 2026 rerun reproduced those results. Further comparisons can
check whether narrower intervals retain the required coverage while
preserving recovery.

## Issues recorded by the studies

1. **#493 -- the chooser's three traced gaps.** The tree rung reads only
numeric columns; a screened pair on a two-valued column crashes the
tensor construction; one shared step budget with a fixed tree depth
undertrains richer rungs. The first two are mechanical fixes; the third
is a design decision (search the tree depth, tune each rung's step count
before ranking).
2. **#443 -- average-effect interval coverage.** The last causal rerun
covered the truth on 5 of 8 replicates, down from 7 of 8.
3. **#478 -- the adjustment-set grade.** The causal battery's structure
score has stayed at 2 across five analyst runs.
4. **#484, #487 -- causal friction from reruns 5 and 6.** Defaults that
ignore the spec, a missing outcome-missing option, internal names in
warnings.
5. **Interval calibration on time-ordered panels.** The PM2.5 80% band
covers 0.74 overall for every library, 0.68 in winter; a time-ordered
calibration helper for the quantile family is the recorded remedy.
6. **The chooser's tolerance on few time-ordered folds.** Two standard
errors across five rolling folds demands roughly a 14% improvement
before the chooser adopts; this threshold can prevent adoption when the
comparison uses a single validation year. The tolerance and fold count need
to be evaluated for that setting.
7. **#499 -- a stale demonstration cell.** The grouped-folds demo in
[](../usage/choosing_terms) raises since the integer-label detection changed
(#441); its `next()` finds no finding and stops. It needs a decision on
what that example should now show.